# Exploratory only

Source of truth = Python modules under `app/` and `scripts/`. These notebooks are retained for EDA, experiment notes, and demo walkthroughs only.

# 📊 Exploratory Data Analysis (EDA) — Aerial Object Detection (OBB)

## Overview

This notebook focuses on the **exploratory data analysis (EDA)** of an aerial imagery dataset annotated with **oriented bounding boxes (OBB)**.

The goal of this step is not to train a model, but to **deeply understand the dataset** before any modeling phase.

In object detection tasks—especially in aerial imagery—EDA is a critical step to identify potential challenges such as:

* small object sizes
* dense object distributions
* class imbalance
* annotation quality

## Objectives

The main objectives of this analysis are:

* Understand the structure of the dataset (images and annotations)
* Verify dataset integrity (missing or inconsistent labels)
* Parse and visualize oriented bounding boxes (OBB)
* Analyze class distribution
* Measure object density per image
* Identify potential challenges for training

## Why EDA Matters

A proper EDA helps to:

* Detect errors in annotations early
* Avoid training on corrupted or biased data
* Guide preprocessing decisions (filtering, tiling, etc.)
* Anticipate model limitations

## Scope of This Notebook

This notebook covers:

1. Dataset loading and extraction
2. Data integrity checks
3. Annotation parsing (DOTA format)
4. Visualization of oriented bounding boxes
5. Preliminary dataset analysis

⚠️ Model training and improvements (tiling, optimization) are handled in separate notebooks.

## Expected Outcome

By the end of this notebook, we will have a **clear understanding of the dataset**, its structure, and its challenges, allowing us to design a more effective detection pipeline in the next stages.


 # I - EDA

## 📊 Exploratory Data Analysis (EDA)

In this section, we perform an exploratory analysis of the DOTA subset dataset.

The objective is to:
- understand the structure of the dataset
- verify the integrity of images and annotations
- visualize oriented bounding boxes (OBB)
- analyze class distribution
- inspect object density per image

This step is critical before training to ensure that:
- annotations are correctly parsed
- bounding boxes are accurate
- the dataset is consistent and usable

We work on a curated subset to enable faster iteration and debugging.

##1 - imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import Counter

plt.rcParams["figure.figsize"] = (8, 8)

##2 — Dézip dataset

In [ ]:
import zipfile

ZIP_PATH = "/content/sample.zip"
EXTRACT_PATH = "/content/data"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset dézippé")

##3 — Chemins dataset

In [ ]:
BASE_PATH = "/content/data/sample"

IMG_DIR = os.path.join(BASE_PATH, "images")
LBL_DIR = os.path.join(BASE_PATH, "labelTxt")

##4 - Vérification dataset

In [ ]:
images = [f for f in os.listdir(IMG_DIR) if f.endswith(".png")]
labels = [f for f in os.listdir(LBL_DIR) if f.endswith(".txt")]

print("Nombre d'images:", len(images))
print("Nombre de labels:", len(labels))

missing_labels = [f for f in images if f.replace(".png", ".txt") not in labels]
print("Images sans label:", len(missing_labels))

 ## 5 - Lecture annotations

In [ ]:
def read_dota_label(file_path):
    objects = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 9:
            continue

        coords = list(map(float, parts[:8]))
        label = parts[8]

        polygon = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]

        objects.append({
            "polygon": polygon,
            "label": label
        })

    return objects

## 6 - Visualisation

In [ ]:
def plot_dota_annotation(img_path, label_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    objects = read_dota_label(label_path)

    for obj in objects:
        pts = [(int(x), int(y)) for x, y in obj["polygon"]]

        # Dessiner le rectangle orienté
        for i in range(4):
            pt1 = pts[i]
            pt2 = pts[(i+1) % 4]
            cv2.line(img, pt1, pt2, (255, 0, 0), 2)

        # Label
        cv2.putText(img, obj["label"], pts[0],
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (0,255,0), 1)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

##7 - Test

In [ ]:
sample_images = random.sample(images, 5)

for f in sample_images:
    img_path = os.path.join(IMG_DIR, f)
    lbl_path = os.path.join(LBL_DIR, f.replace(".png", ".txt"))

    print(f)
    plot_dota_annotation(img_path, lbl_path)

## ✅ EDA Summary

The exploratory analysis confirms that:

- images and annotations are correctly aligned
- oriented bounding boxes are properly parsed and visualized
- the dataset contains multiple object classes with varying frequencies
- object density per image is high, indicating a challenging detection task

Key observations:
- objects appear at arbitrary orientations
- scenes are dense with many small objects
- class distribution is imbalanced

Conclusion:
The dataset is valid and suitable for training an oriented object detection model.

# II - DATASET PREPARATION (FILTERING)

## 🧹 Dataset Preparation and Filtering

After validating the dataset through exploratory analysis, we proceed to prepare the data for training.

The objective of this step is to:
- reduce noise in the dataset
- focus on the most relevant object classes
- simplify the learning task for the model
- improve training stability and performance

Instead of using all available classes, we select a subset of frequently occurring and meaningful categories.

Selected classes:
- plane
- ship
- small-vehicle
- large-vehicle

This filtering step ensures that the dataset is more consistent and better suited for a first training baseline.

##1 - Classes cibles

In [ ]:
TARGET_CLASSES = {
    "plane",
    "ship",
    "small-vehicle",
    "large-vehicle"
}

## ⚙️ Filtering Strategy

For each image:
- we read its annotation file
- we keep only objects belonging to the selected classes
- we discard all other annotations

Images with no remaining valid objects are removed.

This ensures:
- a cleaner dataset
- reduced noise
- better learning signal

## 2 - Filtering dataset

In [ ]:
import os

FILTERED_PATH = "/content/data/filtered"

IMG_OUT = os.path.join(FILTERED_PATH, "images")
LBL_OUT = os.path.join(FILTERED_PATH, "labelTxt")

os.makedirs(IMG_OUT, exist_ok=True)
os.makedirs(LBL_OUT, exist_ok=True)

kept_images = 0
total_objects = 0

for img_file in os.listdir(IMG_DIR):
    if not img_file.endswith(".png"):
        continue

    lbl_file = img_file.replace(".png", ".txt")
    lbl_path = os.path.join(LBL_DIR, lbl_file)

    if not os.path.exists(lbl_path):
        continue

    new_lines = []

    with open(lbl_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 9:
            continue

        label = parts[8]

        if label in TARGET_CLASSES:
            new_lines.append(line)
            total_objects += 1

    # garder uniquement images utiles
    if len(new_lines) > 0:
        kept_images += 1

        os.system(f"cp {os.path.join(IMG_DIR, img_file)} {IMG_OUT}")

        with open(os.path.join(LBL_OUT, lbl_file), "w") as f:
            f.writelines(new_lines)

print("Images gardées:", kept_images)
print("Objets conservés:", total_objects)

## 3 - Vérification

In [ ]:
print("Images filtrées:", len(os.listdir(IMG_OUT)))
print("Labels filtrés:", len(os.listdir(LBL_OUT)))

## 4 - Visualisation rapide

In [ ]:
filtered_images = os.listdir(IMG_OUT)
sample_filtered = random.sample(filtered_images, 3)

for f in sample_filtered:
    img_path = os.path.join(IMG_OUT, f)
    lbl_path = os.path.join(LBL_OUT, f.replace(".png", ".txt"))

    print(f)
    plot_dota_annotation(img_path, lbl_path)

## ✅ Filtered Dataset Summary

After filtering, the dataset contains:

- only relevant object classes
- cleaner and more consistent annotations
- reduced complexity and noise

Benefits:
- faster training
- improved convergence
- better model focus

The dataset is now prepared for the next step: splitting into training and validation sets.

# III -  DATASET SPLITTING

## 🔀 Dataset Splitting

To properly evaluate model performance, the filtered dataset is split into two subsets:

- training set (80%)
- validation set (20%)

This separation ensures that the model is evaluated on unseen data during training, preventing overfitting and providing a more realistic estimate of performance.

In [ ]:
import os, shutil, random

FILTERED_PATH = "/content/data/filtered"
SPLIT_PATH = "/content/data/processed/split"

train_img = os.path.join(SPLIT_PATH, "train/images")
train_lbl = os.path.join(SPLIT_PATH, "train/labelTxt")

val_img = os.path.join(SPLIT_PATH, "val/images")
val_lbl = os.path.join(SPLIT_PATH, "val/labelTxt")

os.makedirs(train_img, exist_ok=True)
os.makedirs(train_lbl, exist_ok=True)
os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

images = [f for f in os.listdir(os.path.join(FILTERED_PATH, "images")) if f.endswith(".png")]

random.shuffle(images)

split_idx = int(0.8 * len(images))

train_files = images[:split_idx]
val_files = images[split_idx:]

print("Train:", len(train_files))
print("Val:", len(val_files))

In [ ]:
for f in train_files:
    shutil.copy(
        os.path.join(FILTERED_PATH, "images", f),
        os.path.join(train_img, f)
    )
    shutil.copy(
        os.path.join(FILTERED_PATH, "labelTxt", f.replace(".png", ".txt")),
        os.path.join(train_lbl, f.replace(".png", ".txt"))
    )

for f in val_files:
    shutil.copy(
        os.path.join(FILTERED_PATH, "images", f),
        os.path.join(val_img, f)
    )
    shutil.copy(
        os.path.join(FILTERED_PATH, "labelTxt", f.replace(".png", ".txt")),
        os.path.join(val_lbl, f.replace(".png", ".txt"))
    )

print("Split terminé")

In [ ]:
print("Train images:", len(os.listdir(train_img)))
print("Val images:", len(os.listdir(val_img)))

In [ ]:
# Vérifier correspondance images/labels

def check_folder(img_dir, lbl_dir):
    imgs = set(os.listdir(img_dir))
    lbls = set(f.replace(".txt", ".png") for f in os.listdir(lbl_dir))

    print("Mismatch:", len(imgs - lbls))

check_folder(train_img, train_lbl)
check_folder(val_img, val_lbl)

In [ ]:
import shutil

shutil.make_archive(
    "/content/processed",          # nom du zip
    'zip',
    "/content/data/processed"      # dossier à zipper
)

print("processed.zip créé")

## ✅ Final Dataset Ready

The dataset is now fully prepared for training:

- cleaned and filtered
- organized into training and validation sets
- consistent and ready for MMRotate

This structured dataset will be used to train an oriented object detection model in the next stage.